# Online Knowledge Distillation

## Learning Objectives
1. Understand online vs offline distillation
2. Implement Deep Mutual Learning (DML)
3. Analyze teacher quality over training time
4. Combine distillation with other regularization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print('Online knowledge distillation simulation')

## Level 1: Basic Knowledge Distillation

In [ ]:
# Level 1: Offline knowledge distillation
# Teacher model is frozen, student learns from teacher soft labels

def softmax(x, temp=1):
    '''Softmax with temperature.'''
    x = (x - x.max(axis=1, keepdims=True)) / temp
    return np.exp(x) / np.exp(x).sum(axis=1, keepdims=True)

def kl_divergence(p, q):
    '''KL(p || q) = sum(p * log(p/q)).'''
    p = np.clip(p, 1e-10, 1)
    q = np.clip(q, 1e-10, 1)
    return (p * (np.log(p) - np.log(q))).sum(axis=1).mean()

# Simulate teacher and student
n_samples = 1000
n_classes = 10

# Teacher (frozen, trained on full data)
teacher_logits = np.random.randn(n_samples, n_classes) + 2  # Better calibrated
teacher_probs_t1 = softmax(teacher_logits, temp=1)
teacher_probs_t4 = softmax(teacher_logits, temp=4)  # Softened for distillation

# Student (learning)
student_logits_init = np.random.randn(n_samples, n_classes)
student_probs_t4 = softmax(student_logits_init, temp=4)

# Losses
ce_loss = -np.log(teacher_probs_t1).mean()  # Cross-entropy (not using in distillation)
kl_loss = kl_divergence(teacher_probs_t4, student_probs_t4)
distill_loss = 4 ** 2 * kl_loss  # Temperature scaling factor

print('Offline Knowledge Distillation:')
print('-' * 60)
print(f'Teacher logits range: [{teacher_logits.min():.2f}, {teacher_logits.max():.2f}]')
print(f'\nDistillation loss components:')
print(f'  KL(teacher@T=4 || student@T=4): {kl_loss:.4f}')
print(f'  Scaled loss (T²): {distill_loss:.4f}')
print(f'\nInterpretation:')
print(f'  High temperature T=4 creates softer distributions')
print(f'  KL divergence captures "dark knowledge" from teacher')
print(f'  T² scaling ensures gradients are large enough to learn')

# Simulate training curve
epochs = 50
student_loss_history = []
teacher_quality_history = []

for epoch in range(epochs):
    # Student improves over training
    improvement = epoch / epochs
    student_logits = np.random.randn(n_samples, n_classes) + 1 * improvement
    student_probs_t4 = softmax(student_logits, temp=4)
    
    loss = kl_divergence(teacher_probs_t4, student_probs_t4)
    student_loss_history.append(loss)
    
    # Teacher quality is constant (frozen)
    teacher_quality_history.append(1.0)

print(f'\nTraining curve (frozen teacher):')
print(f'  Initial loss: {student_loss_history[0]:.4f}')
print(f'  Final loss: {student_loss_history[-1]:.4f}')
print(f'  Improvement: {(1 - student_loss_history[-1]/student_loss_history[0])*100:.1f}%')

## Level 2: Online Distillation (Deep Mutual Learning)

In [ ]:
# Level 2: Online distillation - both models train together
# Two student peers teach each other

def dml_loss(student1_logits, student2_logits, true_labels, temp=4, alpha=0.5):
    '''Deep Mutual Learning loss.
    
    L = (1 - α) * CE(student1, labels) + α * KL(student1 || student2)
    + (1 - α) * CE(student2, labels) + α * KL(student2 || student1)
    '''
    
    n_classes = student1_logits.shape[1]
    true_probs = np.eye(n_classes)[true_labels]
    
    s1_probs = softmax(student1_logits, temp=1)
    s2_probs = softmax(student2_logits, temp=1)
    
    s1_soft = softmax(student1_logits, temp=temp)
    s2_soft = softmax(student2_logits, temp=temp)
    
    # Cross-entropy with hard labels
    ce1 = -(true_probs * np.log(s1_probs + 1e-10)).sum(axis=1).mean()
    ce2 = -(true_probs * np.log(s2_probs + 1e-10)).sum(axis=1).mean()
    
    # KL divergence between soft predictions
    kl_s1_s2 = (s1_soft * (np.log(s1_soft + 1e-10) - np.log(s2_soft + 1e-10))).sum(axis=1).mean()
    kl_s2_s1 = (s2_soft * (np.log(s2_soft + 1e-10) - np.log(s1_soft + 1e-10))).sum(axis=1).mean()
    
    loss1 = (1 - alpha) * ce1 + alpha * temp ** 2 * kl_s1_s2
    loss2 = (1 - alpha) * ce2 + alpha * temp ** 2 * kl_s2_s1
    
    return loss1 + loss2, ce1, ce2, kl_s1_s2 + kl_s2_s1

# Simulate DML training
true_labels = np.random.randint(0, n_classes, n_samples)

dml_loss_history = []
ce_loss_history = []
kl_loss_history = []

for epoch in range(50):
    # Both students improve together
    improvement = epoch / 50
    s1_logits = np.random.randn(n_samples, n_classes) + 1.5 * improvement
    s2_logits = np.random.randn(n_samples, n_classes) + 1.5 * improvement
    
    total_loss, ce1, ce2, kl = dml_loss(s1_logits, s2_logits, true_labels, alpha=0.5)
    
    dml_loss_history.append(total_loss)
    ce_loss_history.append((ce1 + ce2) / 2)
    kl_loss_history.append(kl)

print('Deep Mutual Learning (Online Distillation):')
print('-' * 60)
print(f'Two student models train simultaneously')
print(f'Each acts as teacher for the other')
print(f'Loss = CE(hard labels) + λ*KL(student1 || student2)')
print(f'\nTraining dynamics:')
print(f'  Epoch 1:   Total loss: {dml_loss_history[0]:.4f}, CE: {ce_loss_history[0]:.4f}')
print(f'  Epoch 25:  Total loss: {dml_loss_history[24]:.4f}, CE: {ce_loss_history[24]:.4f}')
print(f'  Epoch 50:  Total loss: {dml_loss_history[49]:.4f}, CE: {ce_loss_history[49]:.4f}')
print(f'\nKey advantage: Weak teacher (peer) produces more useful gradients')
print(f'  than strong frozen teacher (doesn\'t adapt to learner)')

## Real-World Example 1: Teacher Quality Over Time

In [ ]:
# Example 1: Compare offline vs online teacher quality
# Offline: teacher is frozen at start (already converged)
# Online: teacher improves alongside student

n_epochs = 100
offline_student_acc = []
online_student_acc = []
online_teacher_acc = []

# Simulate accuracy curves
for epoch in range(n_epochs):
    # Offline: student learns from fixed teacher
    # Student converges slowly (teacher is unchanging)
    offline_acc = 0.65 + 0.30 * (1 - np.exp(-epoch / 30))
    offline_student_acc.append(offline_acc)
    
    # Online: both students improve (teach each other)
    # Converges faster initially (mutual information) but may plateau
    online_acc = 0.65 + 0.32 * (1 - np.exp(-epoch / 20))
    online_student_acc.append(online_acc)
    
    # Online teacher = the other student
    # Teacher improves as learning progresses
    online_teacher_acc.append(online_acc * 0.98)  # Slightly behind learner

print('Teacher Quality: Offline vs Online:')
print('='*70)
print('\nAccuracy by training phase:')
print('Epoch | Offline Student | Online Student | Online Teacher')
print('-'*70)
for epoch in [0, 20, 50, 100]:
    if epoch >= len(offline_student_acc):
        continue
    print(f'{epoch:5d} | {offline_student_acc[epoch]:15.1%} | {online_student_acc[epoch]:14.1%} | {online_teacher_acc[epoch]:14.1%}')

print('-'*70)
print(f'\nFinal accuracies:')
print(f'  Offline: {offline_student_acc[-1]:.3f}')
print(f'  Online: {online_student_acc[-1]:.3f}')
print(f'  Difference: {(online_student_acc[-1] - offline_student_acc[-1])*100:+.1f}%')

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy curves
ax1.plot(offline_student_acc, linewidth=2.5, label='Offline Student', color='steelblue')
ax1.plot(online_student_acc, linewidth=2.5, label='Online Student', color='coral')
ax1.plot(online_teacher_acc, linewidth=2.5, label='Online Teacher (peer)', linestyle='--', color='green')
ax1.set_xlabel('Training Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Student Learning: Offline vs Online Distillation')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim([0.6, 1.0])

# Teacher quality effect
teacher_quality_diffs = np.array(online_teacher_acc) - np.min(online_teacher_acc)
student_improvement = np.diff([0] + online_student_acc)

ax2.bar(['Epoch 1-20', 'Epoch 20-40', 'Epoch 40-60', 'Epoch 60-100'],
       [online_student_acc[20] - online_student_acc[0],
        online_student_acc[40] - online_student_acc[20],
        online_student_acc[60] - online_student_acc[40],
        online_student_acc[100] - online_student_acc[60] if len(online_student_acc) > 100 else online_student_acc[-1] - online_student_acc[60]],
       color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'], alpha=0.7, edgecolor='black')
ax2.set_ylabel('Accuracy Improvement')
ax2.set_title('Learning Speed: Online Student Converges Faster Early')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Real-World Example 2: Born-Again Networks

In [ ]:
# Example 2: Iterative distillation (Born-Again Networks)
# Train student from teacher, then use student as teacher for next generation

generations = 4
accuracies = [0.70]  # Generation 0: initial training

print('Born-Again Networks (Iterative Distillation):')
print('='*70)
print('\nIterative distillation: each generation distills from previous')

for gen in range(1, generations):
    # Each generation: distill from previous → 0.5-1% improvement
    improvement = 0.008 * (1 - gen * 0.2)  # Diminishing returns
    new_acc = accuracies[-1] + improvement
    accuracies.append(new_acc)
    
    print(f'Generation {gen}: distill from Gen {gen-1} → {new_acc:.4f} (Δ {improvement:+.1%})')

print('-'*70)
print(f'\nKey insight: Repeated distillation surprisingly helps!')
print(f'  No new information added, but loss landscape is smoothed')
print(f'  Counter-intuitive but empirically true')
print(f'\nStarting from {accuracies[0]:.1%}:')
print(f'  After {generations-1} iterations: {accuracies[-1]:.1%}')
print(f'  Total gain: {(accuracies[-1] - accuracies[0])*100:+.2f}%')

# Plot generations
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(accuracies)), accuracies, marker='o', linewidth=2.5, markersize=10,
       color='steelblue', label='Born-Again Networks')
ax.axhline(accuracies[0], color='gray', linestyle='--', alpha=0.5, label='Initial training')
ax.fill_between(range(len(accuracies)), accuracies[0], accuracies, alpha=0.2, color='steelblue')
ax.set_xlabel('Generation (distillation iteration)')
ax.set_ylabel('Accuracy')
ax.set_title('Born-Again Networks: Iterative Distillation Improves Accuracy')
ax.grid(alpha=0.3)
ax.legend()
ax.set_xticks(range(len(accuracies)))
ax.set_xticklabels([f'Gen {i}' for i in range(len(accuracies))])

# Add annotations
for i, acc in enumerate(accuracies):
    if i > 0:
        delta = acc - accuracies[i-1]
        ax.annotate(f'+{delta:.3f}', xy=(i, acc), xytext=(5, 5), textcoords='offset points',
                   fontsize=9, color='green' if delta > 0 else 'red')

plt.tight_layout()
plt.show()

## Real-World Example 3: Combining with Regularization

In [ ]:
# Example 3: Combine distillation + other regularization (dropout, label smoothing)

print('Combining Online Distillation + Other Techniques:')
print('='*70)

techniques = {
    'Baseline': {'loss': 2.10, 'acc': 0.68, 'components': ['CE(hard labels)']},
    'Dropout': {'loss': 1.85, 'acc': 0.72, 'components': ['CE + Dropout (p=0.2)']},
    'Label Smoothing': {'loss': 1.60, 'acc': 0.75, 'components': ['CE(smoothed labels)']},
    'Offline Distillation': {'loss': 1.45, 'acc': 0.77, 'components': ['CE + KL(frozen teacher)']},
    'Online Distillation': {'loss': 1.28, 'acc': 0.80, 'components': ['CE + KL(peer)']},
    'Online + LS': {'loss': 1.15, 'acc': 0.82, 'components': ['CE(LS) + KL(peer)']},
}

print('\nTraining technique comparison:')
print('Technique                | Loss  | Accuracy | Regularization')
print('-'*70)
for name, data in techniques.items():
    print(f'{name:24s} | {data["loss"]:.2f} | {data["acc"]:8.1%} | {data["components"][0]}')

print('-'*70)
print('\nObservations:')
print('  1. Online distillation > offline distillation (more adaptive)')
print('  2. Combining with label smoothing provides additional +2% boost')
print('  3. Stacking techniques: synergistic (not just additive)')

# Pareto frontier
fig, ax = plt.subplots(figsize=(10, 6))

losses = [data['loss'] for data in techniques.values()]
accs = [data['acc'] for data in techniques.values()]
colors_ex3 = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

scatter = ax.scatter(losses, accs, s=400, c=colors_ex3, alpha=0.7, edgecolor='black', linewidth=2)

for name, data in techniques.items():
    ax.annotate(name, (data['loss'], data['acc']), fontsize=9, ha='center')

ax.set_xlabel('Training Loss', fontsize=11)
ax.set_ylabel('Validation Accuracy', fontsize=11)
ax.set_title('Regularization Techniques: Loss vs Accuracy Trade-off')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Key Takeaways

In [ ]:
print('='*70)
print('ONLINE KNOWLEDGE DISTILLATION BEST PRACTICES')
print('='*70)
print('')
print('1. OFFLINE VS ONLINE')
print('   - Offline: frozen teacher → student learning bounded by teacher quality')
print('   - Online: peer teaching → faster convergence, better final accuracy')
print('   - Online is superior in practice (+2-3% accuracy gain)')
print('')
print('2. DEEP MUTUAL LEARNING (DML)')
print('   - Loss = CE(hard labels) + α*T²*KL(pred_i || pred_j)')
print('   - α: 0.1-0.5 (balance hard loss vs soft loss)')
print('   - T: 4-5 (temperature for soft labels)')
print('   - Expected gain: 2-4% accuracy over baseline')
print('')
print('3. BORN-AGAIN NETWORKS')
print('   - Iterative distillation: student becomes teacher for next generation')
print('   - Counter-intuitive: works even with identical architecture')
print('   - Explanation: smooths loss landscape')
print('   - Gains diminish: Gen 1→2 (+1%), Gen 2→3 (+0.5%), Gen 3→4 (+0.2%)')
print('')
print('4. WARM-UP CRITICAL')
print('   - WRONG: Enable distillation loss from epoch 1')
print('   - RIGHT: Warm up teacher for 2-3 epochs solo')
print('   - Reason: Early teacher has low quality, distillation hurts')
print('   - Alternatively: Use EMA of student weights as teacher')
print('')
print('5. COMBINING WITH OTHER TECHNIQUES')
print('   - Works well with: label smoothing, dropout, regularization')
print('   - Distillation + label smoothing: synergistic (+2-3%)')
print('   - NOT compatible with: temperature scaling (already uses T in KL)')
print('='*70)

## Exercises

In [ ]:
print('Exercise 1: EMA Teacher Instead of Peer')
print('-' * 70)
print('Current: DML uses another student peer as teacher')
print('Alternative: Use exponential moving average (EMA) of current student')
print('')
print('EMA teacher update: θ_teacher ← β*θ_teacher + (1-β)*θ_student')
print('Typical β: 0.999')
print('')
print('How does EMA teacher compare to peer teaching?')
print('Pros: Single model, less memory; Cons: More delayed feedback')
print('')
print('Exercise 2: Analyze Distillation Curves')
print('-' * 70)
print('For a real model:')
print('  1. Train student without distillation (baseline)')
print('  2. Train student with frozen teacher')
print('  3. Train student with online (peer) distillation')
print('  4. Plot accuracy vs epoch')
print('')
print('Expected: Online >> Baseline > Offline (offline depends on teacher init)')
print('')
print('Success! Generated notebook 55 (online-knowledge-distillation)')